In [ ]:
!pip install --upgrade git+https://github.com/ariaghora/mpstemmer.git
!pip install Levenshtein

  Cloning https://github.com/ariaghora/mpstemmer.git to /tmp/pip-req-build-_m82qm3t
  Running command git clone --filter=blob:none --quiet https://github.com/ariaghora/mpstemmer.git /tmp/pip-req-build-_m82qm3t
  Resolved https://github.com/ariaghora/mpstemmer.git to commit 25a5fd923af163a7eac3a5ec976984156ca8fa8b
  Preparing metadata (setup.py) ... done
  Created wheel for mpstemmer: filename=mpstemmer-0.1.0-py3-none-any.whl size=99832 sha256=962fd76bee48c3eb71b00d826c27534b836e0ae187f389b90e61b3ce7cf284e8
  Stored in directory: /tmp/pip-ephem-wheel-cache-249fjeg0/wheels/d0/b9/fd/dba0c7d44e5338db51da6ffcbe96db4326db3996a403f5ced1
Successfully built mpstemmer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 15.8 MB/s eta 0:00:00


In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
sw = set(stopwords.words('indonesian'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import string
from mpstemmer import MPStemmer
stemmer = MPStemmer()

def scrape_detik_article(base_url, max_pages=5):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    all_text = []

    for page in range(1, max_pages + 1):
        url = base_url if page == 1 else f"{base_url}?page={page}"
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            break

        soup = BeautifulSoup(response.text, 'html.parser')
        body = soup.find('div', class_='detail__body-text itp_bodycontent')
        if not body:
            break

        paragraphs = body.find_all('p')
        page_text = ' '.join(
            p.get_text().strip() for p in paragraphs
            if 'para_caption' not in p.get('class', [])
        )
        all_text.append(page_text)

        pagination_text = soup.get_text()
        if f"Halaman {page} dari {page}" in pagination_text:
            break

    return ' '.join(all_text)

# ambil langsung dari web
url = "https://news.detik.com/berita/d-8675026/misi-paket-vape-narkoba-berujung-jule-direhabilitasi"
teks_artikel_asli = scrape_detik_article(url)

print(teks_artikel_asli)
print(f"\nJumlah karakter: {len(teks_artikel_asli)}")

def remove_duplicate_text(text):
    n = len(text)
    half = n // 2
    if text[:half].strip() == text[half:].strip():
        return text[:half].strip()
    return text

teks_artikel_asli = remove_duplicate_text(teks_artikel_asli)
print(f"Jumlah karakter setelah menghapus duplikat: {len(teks_artikel_asli)}")

sentences_raw = re.split(r'(?<=[.!?])\s+', teks_artikel_asli.strip())

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    text = stemmer.stem_kalimat(text)  # MPStemmer
    text = " ".join([w for w in text.split() if w not in sw])  # stopwords
    return text

docs = [preprocess(s) for s in sentences_raw]
docs = [d for d in docs if d.strip() != '']
print(f"Jumlah dokumen (kalimat): {len(docs)}")

sentences_tokenized = [d.split() for d in docs]
print(sentences_tokenized[:3])  # cek 3 contoh pertama

Selebgram Julia Prastini atau Jule ditangkap terkait kasus vape narkoba. Dia mendapatkan rehabilitasi setelah kedapatan positif mengonsumsi vape narkoba. Jule ditangkap di sebuah apartemen di Cilandak, Jakarta Selatan (Jaksel), pada Senin (14/9/2026). Ia diamankan saat hendak menerima paket vape narkoba. Dalam penangkapan tersebut, polisi menyita paket berisi 1 catridge vape narkoba dan tiga vape lainnya dalam keadaan kosong sisa pakai. Jule mengaku mengonsumsi narkoba karena stress. Pengungkapan kasus bermula saat anggota Satresnarkoba Polresta Bandara Soekarno-Hatta mengamankan dua wanita berinisial RR dan RN di sebuah apartemen di kawasan Kembangan, Jakarta Barat, pada Senin (14/9/2026) sekitar pukul 13.00 WIB. "Dari lokasi tersebut, petugas menemukan 27 cartridge vape yang diduga mengandung etomidate," ujar Kabid Humas Polda Metro Jaya Kombes Budi Hermanto dalam keterangannya kepada wartawan, Sabtu (19/9). Dari pengungkapan terhadap RR dan RN, polisi kemudian mengamankan pria asal 

**Bag of Word**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

vectorizer_bow = CountVectorizer()  # tanpa binary=True -> ini hitung frekuensi, bukan cuma ada/tidak
X_bow = vectorizer_bow.fit_transform(docs)
df_bow = pd.DataFrame(X_bow.toarray(), columns=vectorizer_bow.get_feature_names_out())
print("Shape BoW:", df_bow.shape)
df_bow

Shape BoW: (33, 136)


,1300,1492026,199,20detik,229,2292026,27,99,akp,alas,...,terang,terima,tim,utara,vape,video,wanita,wartawan,wib,xc
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0
7,0,0,1,0,0,0,1,0,0,0,...,1,0,0,0,1,0,0,1,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
9,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0


**One Hot Encoding**

In [ ]:
vectorizer_ohe = CountVectorizer(binary=True)
X_ohe = vectorizer_ohe.fit_transform(docs)
df_ohe = pd.DataFrame(X_ohe.toarray(), columns=vectorizer_ohe.get_feature_names_out())
print("Shape One-Hot:", df_ohe.shape)
df_ohe

Shape One-Hot: (33, 136)


,1300,1492026,199,20detik,229,2292026,27,99,akp,alas,...,terang,terima,tim,utara,vape,video,wanita,wartawan,wib,xc
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0
7,0,0,1,0,0,0,1,0,0,0,...,1,0,0,0,1,0,0,1,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
9,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0


**TF IDF**

In [ ]:
#TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_tfidf = TfidfVectorizer()
X_tfidf = vectorizer_tfidf.fit_transform(docs)
df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer_tfidf.get_feature_names_out())
print("Shape TF-IDF:", df_tfidf.shape)

print("\nTop 10 kata dengan skor TF-IDF rata-rata tertinggi:")
print(df_tfidf.mean(axis=0).sort_values(ascending=False).head(10))

Shape TF-IDF: (33, 136)

Top 10 kata dengan skor TF-IDF rata-rata tertinggi:
jule            0.120894
menarkoba       0.091705
vape            0.085686
pakai           0.080772
rehabilitasi    0.072571
jp              0.065424
asesmen         0.059078
periksa         0.055513
konsumsi        0.054271
polisi          0.052477
dtype: float64


**Word2vec**

In [ ]:
!pip install gensim
import gensim.models

model_w2v = gensim.models.Word2Vec(sentences_tokenized, vector_size=50, window=5, min_count=1, workers=4)
w2v = model_w2v.wv

print("Jumlah vocabulary:", len(w2v.index_to_key))
for word in ['jule', 'narkoba', 'vape', 'polisi']:
    if word in w2v:
        print(f"\nKata paling mirip dengan '{word}':")
        print(w2v.similar_by_key(word, topn=5))

Jumlah vocabulary: 138

Kata paling mirip dengan 'jule':
[('terima', 0.32559913396835327), ('jaring', 0.27275052666664124), ('sabtu', 0.27228018641471863), ('konsumsi', 0.2720145285129547), ('penyidik', 0.25569960474967957)]

Kata paling mirip dengan 'narkoba':
[('1492026', 0.30427053570747375), ('jp', 0.2940376102924347), ('hubung', 0.2791203558444977), ('komunikasi', 0.2676681876182556), ('saksi', 0.24823498725891113)]

Kata paling mirip dengan 'vape':
[('buah', 0.34271568059921265), ('paket', 0.3283090591430664), ('kharisma', 0.2917182445526123), ('prastin', 0.26536521315574646), ('kembang', 0.255998820066452)]

Kata paling mirip dengan 'polisi':
[('1492026', 0.3248099386692047), ('sisa', 0.28512564301490784), ('rr', 0.258049875497818), ('prastini', 0.2394075095653534), ('budi', 0.23682798445224762)]


In [ ]:
!pip install umap-learn[plot]
!pip install holoviews
!pip install -U ipykernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.2/123.2 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 22.5 MB/s eta 0:00:00


In [ ]:
from umap import UMAP
import plotly.express as px
import pandas as pd
import numpy as np

In [ ]:
# Word2vec Visualization

X = UMAP().fit_transform(w2v.vectors)

df2 = pd.DataFrame(X, columns=['umap1', 'umap2'])
df2['word'] = w2v.index_to_key

fig = px.scatter(df2, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height=800, title_text='Word2Vec Visualization')
fig.show()

**FastText**

In [ ]:
model_ft = gensim.models.FastText(sentences_tokenized, vector_size=50, window=5, min_count=1, workers=4)
ft = model_ft.wv

print("Jumlah vocabulary:", len(ft.index_to_key))
for word in ['jule', 'narkoba', 'vape', 'polisi']:
    print(f"\nKata paling mirip dengan '{word}' (FastText):")
    print(ft.similar_by_key(word, topn=5))

Jumlah vocabulary: 138

Kata paling mirip dengan 'jule' (FastText):
[('julia', 0.43302133679389954), ('dasar', 0.36141979694366455), ('bandara', 0.3479076325893402), ('1', 0.3131549060344696), ('catridge', 0.27803677320480347)]

Kata paling mirip dengan 'narkoba' (FastText):
[('menarkoba', 0.627864420413971), ('satresnarkoba', 0.4431687295436859), ('rangkai', 0.40213754773139954), ('etomidate', 0.39513033628463745), ('narkotika', 0.3566195070743561)]

Kata paling mirip dengan 'vape' (FastText):
[('1300', 0.24656572937965393), ('terang', 0.2431277185678482), ('petugas', 0.2243337333202362), ('minggu', 0.21239089965820312), ('positif', 0.2082924097776413)]

Kata paling mirip dengan 'polisi' (FastText):
[('jp', 0.4311685264110565), ('kombes', 0.3645375967025757), ('isi', 0.34381383657455444), ('catridge', 0.3040369153022766), ('kosong', 0.2897456884384155)]


In [ ]:
X = UMAP().fit_transform(ft.vectors)

df2 = pd.DataFrame(X, columns=['umap1', 'umap2'])
df2['word'] = ft.index_to_key

fig = px.scatter(df2, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height=800, title_text='FastText Visualization')
fig.show()